# 06 · Gerçek Türkçe sınav satırlarıyla Keras CNN–BiLSTM–CTC

Girdi `labels.jsonl` kayıtlarında `image`, `text`, `writer_id` alanları olmalıdır. Yalnız izinli, insan tarafından yazılmış ve öğretmen tarafından düzeltilmiş satırlar kabul edilir. Bölmeler **writer_id bazında ayrıdır**.

In [ ]:
!pip install -q "tensorflow>=2.16,<2.22" scikit-learn jiwer
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/gercek_satir && mkdir -p /content/gercek_satir
!unzip -qo /content/drive/MyDrive/sinav_okur/gercek_satir_verisi.zip -d /content/gercek_satir
!find /content/gercek_satir -name labels.jsonl -print

In [ ]:
import json, random
from pathlib import Path
import numpy as np
import tensorflow as tf
from sklearn.model_selection import GroupShuffleSplit
from jiwer import cer, wer

SEED=42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
ROOT=Path('/content/gercek_satir')
LABELS=next(ROOT.rglob('labels.jsonl'))
BASE=LABELS.parent
kayitlar=[json.loads(x) for x in LABELS.read_text(encoding='utf-8').splitlines() if x.strip()]
zorunlu={'image','text','writer_id'}
assert kayitlar and all(zorunlu <= set(k) for k in kayitlar), 'image/text/writer_id zorunlu'
assert len({k['writer_id'] for k in kayitlar}) >= 5, 'Yazar-ayrık test için en az 5 yazar gerekli'
assert all(Path(k['image']).is_absolute() or (BASE/k['image']).exists() for k in kayitlar), 'Eksik görüntü var'
for k in kayitlar:
    k['image']=str(Path(k['image']) if Path(k['image']).is_absolute() else BASE/k['image'])
print(len(kayitlar),'satır',len({k['writer_id'] for k in kayitlar}),'yazar')

In [ ]:
idx=np.arange(len(kayitlar)); groups=np.array([k['writer_id'] for k in kayitlar])
splitli=all(k.get('split') in {'train','validation','test'} for k in kayitlar)
if splitli:
    # Kaynağın resmî test kümesini asla yeniden bölme veya eşik ayarında kullanma.
    train_idx=idx[np.array([k['split']=='train' for k in kayitlar])]
    val_idx=idx[np.array([k['split']=='validation' for k in kayitlar])]
    test_idx=idx[np.array([k['split']=='test' for k in kayitlar])]
    if not len(val_idx):
        gss_val=GroupShuffleSplit(n_splits=1,test_size=.15,random_state=SEED)
        tr_rel,val_rel=next(gss_val.split(train_idx,groups=groups[train_idx]))
        train_idx,val_idx=train_idx[tr_rel],train_idx[val_rel]
else:
    gss=GroupShuffleSplit(n_splits=1,test_size=.20,random_state=SEED)
    train_idx,test_idx=next(gss.split(idx,groups=groups))
    gss2=GroupShuffleSplit(n_splits=1,test_size=.15,random_state=SEED)
    tr_rel,val_rel=next(gss2.split(train_idx,groups=groups[train_idx]))
    train_idx,val_idx=train_idx[tr_rel],train_idx[val_rel]
assert len(train_idx) and len(val_idx) and len(test_idx), 'train/validation/test boş olamaz'
def writers(ii): return {kayitlar[i]['writer_id'] for i in ii}
assert writers(train_idx).isdisjoint(writers(val_idx)|writers(test_idx))
assert writers(val_idx).isdisjoint(writers(test_idx))
chars=sorted(set(''.join(k['text'] for k in kayitlar)))
c2i={c:i for i,c in enumerate(chars)}; BLANK=len(chars)
MAX_LABEL=max(len(k['text']) for k in kayitlar)
print('split',len(train_idx),len(val_idx),len(test_idx),'chars',len(chars),'max',MAX_LABEL)

In [ ]:
H,W=64,1024
def make_arrays(ii):
    paths=np.array([kayitlar[i]['image'] for i in ii])
    texts=[kayitlar[i]['text'] for i in ii]
    lab=np.zeros((len(ii),MAX_LABEL),np.int32); lengths=np.zeros((len(ii),1),np.int32)
    for n,t in enumerate(texts):
        ids=[c2i[c] for c in t]; lab[n,:len(ids)]=ids; lengths[n]=len(ids)
    return paths,lab,lengths,texts
def read_image(path,label,length):
    raw=tf.io.read_file(path); im=tf.io.decode_image(raw,channels=1,expand_animations=False); im=tf.cast(im,tf.float32)
    shape=tf.cast(tf.shape(im)[:2],tf.float32); scale=tf.minimum(H/shape[0],W/shape[1])
    nh=tf.maximum(1,tf.cast(tf.round(shape[0]*scale),tf.int32)); nw=tf.maximum(1,tf.cast(tf.round(shape[1]*scale),tf.int32))
    im=tf.image.resize(im,[nh,nw]); im=tf.image.pad_to_bounding_box(im,(H-nh)//2,0,H,W)
    im=1.0-im/255.0
    return {'image':im,'label':label,'label_length':length}
def dataset(ii,training=False):
    p,l,ll,_=make_arrays(ii); ds=tf.data.Dataset.from_tensor_slices((p,l,ll))
    if training: ds=ds.shuffle(len(ii),seed=SEED,reshuffle_each_iteration=True)
    return ds.map(read_image,num_parallel_calls=tf.data.AUTOTUNE).batch(16).prefetch(tf.data.AUTOTUNE)
train_ds=dataset(train_idx,True); val_ds=dataset(val_idx); test_ds=dataset(test_idx)

In [ ]:
from tensorflow import keras
L=keras.layers
inp=L.Input((H,W,1),name='image')
x=inp
for filters,pool in [(32,(2,2)),(64,(2,2)),(128,(2,1))]:
    x=L.Conv2D(filters,3,padding='same',use_bias=False)(x); x=L.BatchNormalization()(x); x=L.ReLU()(x)
    x=L.Conv2D(filters,3,padding='same',use_bias=False)(x); x=L.BatchNormalization()(x); x=L.ReLU()(x); x=L.MaxPool2D(pool)(x)
x=L.Permute((2,1,3))(x); x=L.Reshape((W//4,(H//8)*128))(x)
x=L.Bidirectional(L.LSTM(256,return_sequences=True,dropout=.2))(x)
x=L.Bidirectional(L.LSTM(256,return_sequences=True,dropout=.2))(x)
out=L.Dense(len(chars)+1,activation='softmax',name='ctc_probs')(x)
inference=keras.Model(inp,out,name='turkce_gercek_satir_ctc')

class CtcTrainer(keras.Model):
    def __init__(self,net): super().__init__(); self.net=net; self.loss_tracker=keras.metrics.Mean(name='loss')
    @property
    def metrics(self): return [self.loss_tracker]
    def _loss(self,batch,training):
        probs=self.net(batch['image'],training=training)
        ilen=tf.fill([tf.shape(probs)[0],1],tf.shape(probs)[1])
        return tf.reduce_mean(keras.backend.ctc_batch_cost(batch['label'],probs,ilen,batch['label_length']))
    def train_step(self,batch):
        with tf.GradientTape() as tape: loss=self._loss(batch,True)
        grads=tape.gradient(loss,self.net.trainable_variables); self.optimizer.apply_gradients(zip(grads,self.net.trainable_variables)); self.loss_tracker.update_state(loss); return {'loss':self.loss_tracker.result()}
    def test_step(self,batch):
        loss=self._loss(batch,False); self.loss_tracker.update_state(loss); return {'loss':self.loss_tracker.result()}
trainer=CtcTrainer(inference); trainer.compile(optimizer=keras.optimizers.Adam(3e-4))
inference.summary()
callbacks=[keras.callbacks.ModelCheckpoint('/content/en_iyi_ctc.weights.h5',save_weights_only=True,save_best_only=True,monitor='val_loss'),keras.callbacks.ReduceLROnPlateau(monitor='val_loss',factor=.5,patience=2),keras.callbacks.EarlyStopping(monitor='val_loss',patience=6,restore_best_weights=True)]
trainer.fit(train_ds,validation_data=val_ds,epochs=60,callbacks=callbacks)

In [ ]:
inference.load_weights('/content/en_iyi_ctc.weights.h5')
def greedy(probs):
    sonuc=[]
    for prob,row in zip(probs,probs.argmax(-1)):
        prev=None; s=[]; conf=[]
        for t,i in enumerate(row):
            i=int(i)
            if i != BLANK and i != prev: s.append(chars[i]); conf.append(float(prob[t,i]))
            prev=i
        sonuc.append((''.join(s).strip(),float(np.mean(conf)) if conf else 0.0))
    return sonuc
test_paths,_,_,refs=make_arrays(test_idx)
tahminler=[]
for batch in test_ds: tahminler.extend(greedy(inference.predict(batch['image'],verbose=0)))
hyps=[x[0] for x in tahminler]; guvenler=np.array([x[1] for x in tahminler])
assert len(hyps)==len(refs)
en_iyi=None
for esik in np.linspace(0.50,0.999,500):
    sec=np.where(guvenler>=esik)[0]
    if len(sec) and cer([refs[i] for i in sec],[hyps[i] for i in sec]) <= .01:
        aday={'threshold':float(esik),'coverage':float(len(sec)/len(refs)),'accepted_cer':float(cer([refs[i] for i in sec],[hyps[i] for i in sec]))}
        if en_iyi is None or aday['coverage']>en_iyi['coverage']: en_iyi=aday
metrics={'cer':float(cer(refs,hyps)),'wer':float(wer(refs,hyps)),'writer_disjoint':True,'test_writers':len(writers(test_idx)),'test_lines':len(refs),'auto_accept_99':en_iyi}
print(metrics)
for r,h in list(zip(refs,hyps))[:20]: print('GT:',r,'\nPR:',h,'\n')
inference.save('/content/turkce_gercek_satir_ctc.keras')
Path('/content/ctc_etiketler.json').write_text(json.dumps(chars,ensure_ascii=False,indent=2),encoding='utf-8')
Path('/content/ctc_metrikleri.json').write_text(json.dumps(metrics,ensure_ascii=False,indent=2),encoding='utf-8')
splits={'train':sorted(writers(train_idx)),'validation':sorted(writers(val_idx)),'test':sorted(writers(test_idx))}
Path('/content/writer_splits.json').write_text(json.dumps(splits,ensure_ascii=False,indent=2),encoding='utf-8')

In [ ]:
import shutil
from google.colab import files
OUT=Path('/content/drive/MyDrive/sinav_okur/modeller/gercek_satir_ctc'); OUT.mkdir(parents=True,exist_ok=True)
for ad in ['turkce_gercek_satir_ctc.keras','ctc_etiketler.json','ctc_metrikleri.json','writer_splits.json']:
    shutil.copy2(Path('/content',ad),OUT/ad)
arsiv=shutil.make_archive('/content/turkce_gercek_satir_ctc','zip',root_dir=OUT)
files.download(arsiv)
print('Drive ve ZIP kaydı:',OUT,arsiv)